# 03 — Training

We fine-tune Qwen2.5-0.5B (student) on the 548 CoT traces distilled from DeepSeek-R1-14B (teacher).

This is a **full fine-tune** all 494M parameters are updated. No LoRA or frozen layers.
At 0.5B, the model fits comfortably in 16GB VRAM with room to spare for large batch sizes.

### What this notebook does
1. Inspect the student model architecture
2. Load the formatted dataset
3. Configure and run SFTTrainer
4. Save the final model checkpoint

## Model Architecture

In [1]:
# Inspect model architecture
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
print(model)
print(f"\nTotal params:     {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## Imports and Setup

We use:
- `transformers` — load the pretrained Qwen2.5-0.5B model and tokenizer
- `trl` — SFTTrainer, a fine-tuning wrapper built specifically for LLMs
- `datasets` — load our formatted train/val splits
- `wandb` — experiment tracking, logs loss curves and eval metrics in real time

In [2]:
import torch
import wandb
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk

# Verify CUDA is available
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"Device:          {torch.cuda.get_device_name(0)}")
print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available:  True
Device:          NVIDIA GeForce RTX 5070 Ti
VRAM:            16.6 GB


## Load Dataset

We load the HuggingFace Dataset saved in `02_dataset_formatting.ipynb`.

- **Train**: 548 examples
- **Val**: 29 examples

Each example has a `text` field formatted as:
```
<|user|>
{math problem}
<|assistant|>
{CoT trace with <think> and <answer> tags}
```

In [3]:
train_ds = load_from_disk("data/processed/train")
val_ds = load_from_disk("data/processed/val")

print(f"Train: {len(train_ds)} examples")
print(f"Val:   {len(val_ds)} examples")
print(f"\nSample:\n{train_ds[0]['text'][:300]}")

Train: 548 examples
Val:   29 examples

Sample:
<|user|>
Simplify $2a(2a^2 + a) - a^2$.
<|assistant|>
Step-by-step explanation:

We start with the expression:
\[2a(2a^2 + a) - a^2\]

First, distribute the \(2a\) across the terms inside the parentheses:
\[
2a \cdot 2a^2 = 4a^3
\]
\[
2a \cdot a = 2a^2
\]

So, after distributing, we have:
\[
4a^3 + 


## Load Model and Tokenizer

We load `Qwen/Qwen2.5-0.5B` in **bf16** (bfloat16) precision.

bf16 vs fp32:
- fp32: full precision, 4 bytes per param → ~2GB for 0.5B
- bf16: half precision, 2 bytes per param → ~1GB for 0.5B
- bf16 trains as stably as fp32 on modern GPUs and is twice as memory efficient

`device_map="cuda"` moves the entire model to your RTX 5070 Ti automatically.

In [4]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" ## padding for either left or right. The model reads left to right 
## With left padding, the model would see padding first and the real content later, which confuses the attention patterns during training,
##Left padding is used during inference/generation — so the model's last token before generating is always a real token, not a PAD.
##Since you're in the training notebook right now, padding_side = 
##"right" is the correct setting. When you get to the evaluation notebook and run inference, you'd flip it to "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda"
)

print(f"Model loaded — dtype: {next(model.parameters()).dtype}")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded — dtype: torch.bfloat16
VRAM used: 0.99 GB


## Initialize W&B

Weights & Biases tracks all training metrics in real time:
- Training loss per step
- Validation loss per eval step
- Learning rate schedule
- GPU utilization

All runs are logged under the `tiny-math-reasoner` project on W&B dashboard.

In [5]:
# Run this to see trace length distribution
from datasets import load_from_disk
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
train_ds = load_from_disk("data/processed/train")

lengths = [len(tokenizer(x['text'])['input_ids']) for x in train_ds]

print(f"Min:    {min(lengths)}")
print(f"Max:    {max(lengths)}")
print(f"Mean:   {sum(lengths)/len(lengths):.0f}")
print(f"Median: {sorted(lengths)[len(lengths)//2]}")
print(f"95th percentile: {sorted(lengths)[int(len(lengths)*0.95)]}")

Min:    27
Max:    895
Mean:   219
Median: 200
95th percentile: 448


## A Note on Token Limits

Three different token numbers appear across this project and they are all different things:

### `num_predict: 2048` — Teacher generation ceiling
This is the maximum number of tokens DeepSeek-R1-14B is **allowed** to generate
per trace. It is a ceiling, not a target. We set it high to ensure the teacher
never gets cut off mid-reasoning on harder problems. In `01_trace_generation.ipynb`

### 895 — Longest trace actually generated
In practice, the teacher never needed anywhere near 2048 tokens. The longest
trace in our dataset was 895 tokens, with a mean of just 219 tokens. This makes
sense — Levels 1–3 problems are relatively simple, and once the model reaches
an `<answer>` tag it stops naturally.

### `max_seq_length: 1024` — Student training window
This is the maximum sequence length the student model (Qwen2.5-0.5B) sees
during training. We set this based on the **95th percentile** of our actual
trace lengths (448 tokens), rounded up to 512. Setting it to 2048
would waste memory and slow down training with no benefit, since almost all
traces fit comfortably within 512 tokens.

### Summary

| Value | What it controls | Why |
|---|---|---|
| 2048 | Teacher max output (Ollama) | Prevent cutoffs on hard problems |
| 895 | Longest trace generated | Reflects actual problem difficulty |
| 1024 | Student training sequence length | Covers 95th percentile of trace lengths |

In [6]:
# w&b init
wandb.init(
    project="tiny-math-reasoner",
    name="qwen05b-full-finetune-run1",
    config={
        "model": MODEL_ID,
        "train_samples": len(train_ds),
        "val_samples": len(val_ds),
        "epochs": 5,
        "batch_size": 8,
        "grad_accum": 4,
        "effective_batch_size": 32,
        "learning_rate": 2e-5,
        "precision": "bf16",
        "packing": True,
        "max_seq_length": 1024,
    }
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## SFTConfig — Training Hyperparameters

Key decisions:

**Batch size + gradient accumulation**
- `per_device_train_batch_size=8` — 8 examples per GPU step
- `gradient_accumulation_steps=4` — accumulate gradients over 4 steps
- Effective batch size = 8 × 4 = **32** — larger batches = more stable gradients

**Learning rate**
- `2e-5` — standard for full fine-tuning of small LLMs
- Too high → catastrophic forgetting of pretrained knowledge
- Too low → slow convergence, underfitting

**Cosine LR scheduler**
- Starts at `2e-5`, warms up for 5% of steps, then decays smoothly to 0
- Prevents sharp loss spikes at the end of training

**Packing**
- Combines multiple short examples into one sequence up to `max_seq_length`
- Eliminates wasted padding tokens → much more efficient training
- Critical when your dataset is small (548 examples)

**Epochs**
- 5 epochs over 548 examples is conservative and safe
- At this scale overfitting is possible beyond 5 epochs

In [14]:
args = SFTConfig(
    # Output
    output_dir="./checkpoints",

    # Training duration
    num_train_epochs=5,

    # Batch size
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,

    # Optimizer
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps= 50,
    weight_decay=0.01,

    # Precision
    bf16=True,

    # Packing
    packing=False,
    max_length=512,

    # Logging and eval
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,

    # W&B
    report_to="none",

    # Dataset
    dataset_text_field="text",
)

## Run Training

SFTTrainer handles:
- Tokenizing and packing dataset automatically
- The training loop that is forward pass, loss, backward pass, optimizer step
- Evaluation on the val set every 50 steps
- Saving checkpoints and loading the best one at the end

In [15]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/548 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/548 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/29 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/29 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.311077,0.380398,0.373830,335177.000000,0.887422


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: WARNING URL not available in offline run


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=90, training_loss=0.33331233130560983, metrics={'train_runtime': 105.2616, 'train_samples_per_second': 26.03, 'train_steps_per_second': 0.855, 'total_flos': 1637937888677376.0, 'train_loss': 0.33331233130560983, 'epoch': 5.0})

In [10]:
import torch
print(f"VRAM allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

VRAM allocated: 6.45 GB


In [17]:
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")
print("Model saved to ./final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./final_model
